# Retail Orders: Data Profile and Quality Gate

**Dataset:** `retail-orders-raw.csv` (commerce: learning-product orders) with `retail-data-dictionary.csv` as the rulebook.
**Decision owner:** Head of Revenue Operations. They decide which products to push, how deep discounts go, and whether payment or refund leakage needs action.
**Run it:** Run all cells top to bottom (Python 3, pandas, numpy). Set the `AS_OF` environment variable to override the as-of date.

| Section | What it does |
|---|---|
| 1 | Load and eyeball the raw data |
| 2 | Executable checks: completeness, uniqueness, validity, consistency, freshness |
| 3 | Gate decision (PASS / WARN / BLOCK) on the raw feed |
| 4 | Remediation: normalize, de-duplicate, quarantine |
| 5 | Re-run the same checks on the certified layer |
| 6 | KPIs computed from certified data only |
| 7 | Reconciliation and exports |

## 1. Load and look

In [1]:
import os, json
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

RAW_PATH = "retail-orders-raw.csv"
DICT_PATH = "retail-data-dictionary.csv"
OUT_DIR = "profile_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# As-of date for freshness and future-date checks. Use pd.Timestamp.today().normalize() in production.
AS_OF = pd.Timestamp(os.environ.get("AS_OF", "2026-11-13"))
MIN_DATE = pd.Timestamp("2025-01-01")          # from the data dictionary
MAX_AGE_DAYS = 1                               # freshness SLA: daily feed

SEGMENTS = ["Student", "Fresher", "Professional"]
CATEGORIES = ["Learning Kit", "Course Access", "Mentor Session"]
STATUSES = ["Paid", "Pending", "Failed", "Refunded"]
RAW_COLS = ["order_id", "order_date", "customer_segment", "city", "category",
            "quantity", "unit_price", "discount_pct", "payment_status"]

dictionary = pd.read_csv(DICT_PATH)
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)       # read everything as text, never let pandas guess
raw = raw.apply(lambda s: s.str.strip()).replace("", np.nan)         # blank cells become real nulls
raw.insert(0, "source_line", raw.index + 2)                          # line number in the CSV file (header is line 1)

display(dictionary)
display(raw)

,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normalization
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only after justification
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normalization


,source_line,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,2,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10,Paid
1,3,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0,paid
2,4,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,5,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
4,6,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5,Paid
5,7,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0,Failed
6,8,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10,Paid
7,9,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105,Paid
8,10,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0,Pending
9,11,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0,Paid


In [2]:
profile = pd.DataFrame({
    "non_null": raw[RAW_COLS].notna().sum(),
    "nulls": raw[RAW_COLS].isna().sum(),
    "distinct": raw[RAW_COLS].nunique(),
})
print(f"{len(raw)} rows x {len(RAW_COLS)} columns")
display(profile)
for c in ["customer_segment", "category", "payment_status"]:
    print(f"\n{c}:", raw[c].value_counts(dropna=False).to_dict())

12 rows x 9 columns


,non_null,nulls,distinct
order_id,12,0,11
order_date,11,1,10
customer_segment,12,0,4
city,11,1,7
category,12,0,3
quantity,12,0,5
unit_price,12,0,3
discount_pct,11,1,5
payment_status,12,0,5



customer_segment: {'Student': 4, 'Professional': 4, 'Fresher': 3, 'student': 1}

category: {'Learning Kit': 5, 'Course Access': 4, 'Mentor Session': 3}

payment_status: {'Paid': 7, 'Pending': 2, 'paid': 1, 'Failed': 1, 'Refunded': 1}


## 2. Executable checks

Each check returns a pass/fail per row. A check **fails** when its pass rate drops below its threshold.
Validity rules judge populated values only, so a blank is counted once (under completeness) and not twice.

In [3]:
def run_checks(df: pd.DataFrame, label: str) -> pd.DataFrame:
    d = df.reset_index(drop=True)
    ident = d["order_id"].fillna("line " + d["source_line"].astype(str))
    qty = pd.to_numeric(d["quantity"], errors="coerce")
    price = pd.to_numeric(d["unit_price"], errors="coerce")
    disc = pd.to_numeric(d["discount_pct"], errors="coerce")
    dt = pd.to_datetime(d["order_date"], format="%Y-%m-%d", errors="coerce")     # strict ISO only
    rows = []

    def add(cid, dim, col, rule, thr, sev, passed):
        passed = passed.astype(bool)
        failed = ident[~passed].tolist()
        rows.append(dict(check_id=cid, dimension=dim, column=col, rule=rule, rows_tested=len(d),
                         rows_failed=len(failed), threshold_pct=thr, severity=sev,
                         failing_orders=", ".join(failed), observed=""))

    # --- Completeness -------------------------------------------------------------
    for cid, col, thr, sev in [("COMP-01", "order_id", 100, "critical"), ("COMP-02", "order_date", 98, "major"),
                               ("COMP-03", "customer_segment", 98, "major"), ("COMP-04", "city", 98, "major"),
                               ("COMP-05", "quantity", 100, "critical"), ("COMP-06", "discount_pct", 98, "minor")]:
        add(cid, "Completeness", col, f"{col} is populated", thr, sev, d[col].notna())

    # --- Uniqueness ---------------------------------------------------------------
    add("UNQ-01", "Uniqueness", "order_id", "order_id appears once", 100, "critical",
        ~d["order_id"].duplicated(keep="first") | d["order_id"].isna())
    add("UNQ-02", "Uniqueness", "all columns", "No exact duplicate rows", 100, "major",
        ~d.duplicated(subset=RAW_COLS, keep="first"))

    # --- Validity (populated values only) ----------------------------------------
    add("VAL-01", "Validity", "order_date", "ISO date (YYYY-MM-DD) between 2025-01-01 and as-of date", 98, "major",
        d["order_date"].isna() | (dt.notna() & (dt >= MIN_DATE) & (dt <= AS_OF)))
    add("VAL-02", "Validity", "quantity", "Whole number greater than zero", 100, "critical",
        d["quantity"].isna() | (qty.notna() & (qty > 0) & (qty == qty.round())))
    add("VAL-03", "Validity", "unit_price", "Non-negative INR amount", 100, "critical",
        d["unit_price"].isna() | (price.notna() & (price >= 0)))
    add("VAL-04", "Validity", "discount_pct", "Between 0 and 100", 100, "critical",
        d["discount_pct"].isna() | (disc.notna() & disc.between(0, 100)))
    add("VAL-05", "Validity", "category", "One of Learning Kit, Course Access, Mentor Session", 100, "major",
        d["category"].isna() | d["category"].isin(CATEGORIES))
    add("VAL-06", "Validity", "customer_segment", "Student, Fresher or Professional after normalization", 98, "minor",
        d["customer_segment"].isna() | d["customer_segment"].str.title().isin(SEGMENTS))
    add("VAL-07", "Validity", "payment_status", "Paid, Pending, Failed or Refunded after normalization", 98, "major",
        d["payment_status"].isna() | d["payment_status"].str.capitalize().isin(STATUSES))

    # --- Consistency --------------------------------------------------------------
    add("CON-01", "Consistency", "customer_segment", "Canonical casing (Student / Fresher / Professional)", 95, "minor",
        d["customer_segment"].isna() | d["customer_segment"].isin(SEGMENTS))
    add("CON-02", "Consistency", "payment_status", "Canonical casing (Paid / Pending / Failed / Refunded)", 95, "minor",
        d["payment_status"].isna() | d["payment_status"].isin(STATUSES))
    mode_price = price.groupby(d["category"]).transform(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
    add("CON-03", "Consistency", "unit_price", "One list price per category", 100, "major",
        price.isna() | mode_price.isna() | (price == mode_price))
    add("CON-04", "Consistency", "order_date", "Single date format (YYYY-MM-DD) across the file", 98, "major",
        d["order_date"].isna() | d["order_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False).astype(bool))

    # --- Freshness ----------------------------------------------------------------
    add("FRS-02", "Freshness", "order_date", "No order dated after the as-of date", 100, "critical", ~(dt > AS_OF))
    latest = dt.max()
    age = (AS_OF - latest).days if pd.notna(latest) else 10**6
    rows.append(dict(check_id="FRS-01", dimension="Freshness", column="order_date",
                     rule=f"Latest order is at most {MAX_AGE_DAYS} day old at the as-of date", rows_tested=1,
                     rows_failed=int(age > MAX_AGE_DAYS), threshold_pct=100, severity="major", failing_orders="",
                     observed=f"latest order {latest.date()} is {age} days before {AS_OF.date()}"))

    res = pd.DataFrame(rows)
    res["pass_rate_pct"] = (100 * (1 - res.rows_failed / res.rows_tested)).round(1)
    res["status"] = np.where(100 * (1 - res.rows_failed / res.rows_tested) >= res.threshold_pct, "PASS", "FAIL")
    order = ["COMP", "UNQ", "VAL", "CON", "FRS"]
    res["_o"] = res.check_id.str[:4].map({k: i for i, k in enumerate(order)})
    res = res.sort_values(["_o", "check_id"]).drop(columns="_o").reset_index(drop=True)
    res.insert(0, "layer", label)
    return res


def gate(res: pd.DataFrame) -> str:
    failed = res[res.status == "FAIL"]
    if (failed.severity == "critical").any():
        return "BLOCK"
    if (failed.severity == "major").any():
        return "WARN"
    return "PASS"


raw_results = run_checks(raw, "raw")
display(raw_results[["check_id", "dimension", "column", "rows_failed", "rows_tested", "pass_rate_pct",
                     "threshold_pct", "severity", "status", "failing_orders"]])

,check_id,dimension,column,rows_failed,rows_tested,pass_rate_pct,threshold_pct,severity,status,failing_orders
0,COMP-01,Completeness,order_id,0,12,100.0,100,critical,PASS,
1,COMP-02,Completeness,order_date,1,12,91.7,98,major,FAIL,RT-1011
2,COMP-03,Completeness,customer_segment,0,12,100.0,98,major,PASS,
3,COMP-04,Completeness,city,1,12,91.7,98,major,FAIL,RT-1005
4,COMP-05,Completeness,quantity,0,12,100.0,100,critical,PASS,
5,COMP-06,Completeness,discount_pct,1,12,91.7,98,minor,FAIL,RT-1003
6,CON-01,Consistency,customer_segment,1,12,91.7,95,minor,FAIL,RT-1003
7,CON-02,Consistency,payment_status,1,12,91.7,95,minor,FAIL,RT-1002
8,CON-03,Consistency,unit_price,0,12,100.0,100,major,PASS,
9,CON-04,Consistency,order_date,1,12,91.7,98,major,FAIL,RT-1002


## 3. Gate decision on the raw feed

In [4]:
raw_gate = gate(raw_results)
by_dim = raw_results.groupby("dimension").apply(
    lambda g: pd.Series({"checks": len(g), "failed": int((g.status == "FAIL").sum())}), include_groups=False)
display(by_dim)
print("RAW FEED GATE:", raw_gate)
print(raw_results.loc[raw_results.status == "FAIL", "observed"].replace("", np.nan).dropna().to_string(index=False))

,checks,failed
dimension,,
Completeness,6,3
Consistency,4,3
Freshness,2,1
Uniqueness,2,2
Validity,7,3


RAW FEED GATE: BLOCK
latest order 2026-01-18 is 299 days before 2026-11-13


## 4. Remediation

Rules applied, in order. Nothing is guessed silently: every change lands in the remediation log.

1. Exact duplicate rows are dropped (first copy kept).
2. Segment and payment status are normalized (trim, case).
3. Dates in `DD/MM/YYYY` are re-formatted to ISO. Justification: the source is India-based and the file is January data, so `03/01/2026` read as 3 Jan sits between RT-1001 (3 Jan) and RT-1003 (5 Jan), matching the order sequence. A month-first reading (1 Mar) would break that sequence.
4. Missing `discount_pct` becomes 0 and is flagged `discount_imputed`. Justification: the order is unsettled and sibling orders without a discount carry an explicit 0. This stays an assumption for the owner to confirm.
5. Anything else that breaks a dictionary rule is **quarantined**, not repaired. Free text such as `two` is not converted to a number.

In [5]:
c = raw.copy()
log, reasons = [], {i: [] for i in c.index}

def note(idx, field, before, after, action, why):
    log.append(dict(source_line=int(raw.loc[idx, "source_line"]), order_id=raw.loc[idx, "order_id"], field=field,
                    before=before, after=after, action=action, justification=why))

# 1. exact duplicates
dup_mask = c.duplicated(subset=RAW_COLS, keep="first")
for i in c[dup_mask].index:
    note(i, "row", "duplicate row", "dropped", "drop", "Exact copy of an earlier row (all 9 columns identical)")
c = c[~dup_mask].copy()
for i in c[c.order_id.duplicated(keep=False) & c.order_id.notna()].index:
    reasons[i].append("conflicting duplicate order_id")

# 2. normalize categorical casing
for col, fn in [("customer_segment", str.title), ("payment_status", str.capitalize)]:
    new = c[col].map(lambda v: fn(v) if isinstance(v, str) else v)
    for i in c.index[(new != c[col]) & c[col].notna()]:
        note(i, col, c.at[i, col], new[i], "normalize", "Case/whitespace normalization per data dictionary")
    c[col] = new

# 3. dates
iso = pd.to_datetime(c.order_date, format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(c.order_date, format="%d/%m/%Y", errors="coerce")
c["date_reformatted"] = iso.isna() & dmy.notna()
for i in c.index[c.date_reformatted]:
    note(i, "order_date", c.at[i, "order_date"], dmy[i].date().isoformat(), "reformat",
         "Read as DD/MM/YYYY; fits the order_id sequence (between 3 Jan and 5 Jan)")
c["order_dt"] = iso.fillna(dmy)

# 4. discount
c["discount_num"] = pd.to_numeric(c.discount_pct, errors="coerce")
c["discount_imputed"] = c.discount_pct.isna()
for i in c.index[c.discount_imputed]:
    note(i, "discount_pct", "(blank)", "0", "impute", "Assumed no discount; flagged for owner confirmation")
c["discount_num"] = c.discount_num.fillna(0)

# 5. quarantine rules
c["qty_num"] = pd.to_numeric(c.quantity, errors="coerce")
c["price_num"] = pd.to_numeric(c.unit_price, errors="coerce")
for i, r in c.iterrows():
    if pd.isna(r.order_id): reasons[i].append("order_id missing")
    if pd.isna(r.order_dt):
        reasons[i].append("order_date missing" if pd.isna(r.order_date) else f"order_date invalid ({r.order_date})")
    elif not (MIN_DATE <= r.order_dt <= AS_OF): reasons[i].append("order_date out of range")
    if pd.isna(r.city): reasons[i].append("city missing")
    if pd.isna(r.customer_segment) or r.customer_segment not in SEGMENTS: reasons[i].append("customer_segment invalid")
    if pd.isna(r.category) or r.category not in CATEGORIES: reasons[i].append("category invalid")
    if pd.isna(r.payment_status) or r.payment_status not in STATUSES: reasons[i].append("payment_status invalid")
    if pd.isna(r.qty_num): reasons[i].append(f"quantity not numeric ({r.quantity})")
    elif r.qty_num <= 0 or r.qty_num != round(r.qty_num): reasons[i].append(f"quantity not a positive whole number ({r.quantity})")
    if pd.isna(r.price_num) or r.price_num < 0: reasons[i].append("unit_price invalid")
    if not (0 <= r.discount_num <= 100): reasons[i].append(f"discount_pct out of range ({r.discount_pct})")

q_idx = [i for i in c.index if reasons[i]]
quarantine = raw.loc[q_idx].copy()
quarantine["quarantine_reason"] = ["; ".join(reasons[i]) for i in q_idx]
for i in q_idx:
    note(i, "row", "failed rules", "quarantined", "quarantine", "; ".join(reasons[i]))

ok = c.drop(index=q_idx)
certified = pd.DataFrame({
    "order_id": ok.order_id, "order_date": ok.order_dt.dt.strftime("%Y-%m-%d"),
    "customer_segment": ok.customer_segment, "city": ok.city, "category": ok.category,
    "quantity": ok.qty_num.astype(int), "unit_price": ok.price_num.astype(int),
    "discount_pct": ok.discount_num, "payment_status": ok.payment_status,
    "discount_imputed": ok.discount_imputed, "date_reformatted": ok.date_reformatted,
    "source_line": ok.source_line,
}).reset_index(drop=True)

remediation_log = pd.DataFrame(log)
print("Certified orders:"); display(certified)
print("Quarantined rows:"); display(quarantine)
print("Remediation log:"); display(remediation_log)

Certified orders:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,discount_imputed,date_reformatted,source_line
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid,False,False,2
1,RT-1002,2026-01-03,Fresher,Bengaluru,Course Access,1,1499,0.0,Paid,False,True,3
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1,1499,0.0,Pending,True,False,4
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,False,False,5
4,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid,False,False,11
5,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15.0,Refunded,False,False,12


Quarantined rows:


,source_line,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,quarantine_reason
5,7,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0,Failed,city missing
6,8,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10,Paid,order_date invalid (2026-13-10); quantity not a positive whole number (-1)
7,9,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105,Paid,discount_pct out of range (105)
8,10,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0,Pending,quantity not numeric (two)
11,13,RT-1011,NaN,Student,Kochi,Mentor Session,1,999,0,Paid,order_date missing


Remediation log:


,source_line,order_id,field,before,after,action,justification
0,6,RT-1004,row,duplicate row,dropped,drop,Exact copy of an earlier row (all 9 columns identical)
1,4,RT-1003,customer_segment,student,Student,normalize,Case/whitespace normalization per data dictionary
2,3,RT-1002,payment_status,paid,Paid,normalize,Case/whitespace normalization per data dictionary
3,3,RT-1002,order_date,03/01/2026,2026-01-03,reformat,Read as DD/MM/YYYY; fits the order_id sequence (between 3 Jan and 5 Jan)
4,4,RT-1003,discount_pct,(blank),0,impute,Assumed no discount; flagged for owner confirmation
5,7,RT-1005,row,failed rules,quarantined,quarantine,city missing
6,8,RT-1006,row,failed rules,quarantined,quarantine,order_date invalid (2026-13-10); quantity not a positive whole number (-1)
7,9,RT-1007,row,failed rules,quarantined,quarantine,discount_pct out of range (105)
8,10,RT-1008,row,failed rules,quarantined,quarantine,quantity not numeric (two)
9,13,RT-1011,row,failed rules,quarantined,quarantine,order_date missing


## 5. Re-run the same checks on the certified layer

In [6]:
cert_text = certified[RAW_COLS + ["source_line"]].astype(str)
cert_results = run_checks(cert_text, "certified")
cert_gate = gate(cert_results)
display(cert_results.loc[cert_results.status == "FAIL",
        ["check_id", "dimension", "rule", "pass_rate_pct", "threshold_pct", "severity", "status", "observed"]])
print("RAW GATE:", raw_gate, "| CERTIFIED GATE:", cert_gate)

,check_id,dimension,rule,pass_rate_pct,threshold_pct,severity,status,observed
10,FRS-01,Freshness,Latest order is at most 1 day old at the as-of date,0.0,100,major,FAIL,latest order 2026-01-18 is 299 days before 2026-11-13


RAW GATE: BLOCK | CERTIFIED GATE: WARN


## 6. KPIs from certified data

Net value = quantity x unit_price x (1 - discount_pct/100). Only certified orders count. Full definitions live in `retail-kpi-dictionary.xlsx`.

In [7]:
k = certified.copy()
k["gross"] = k.quantity * k.unit_price
k["net"] = k.gross * (1 - k.discount_pct / 100)
paid, refunded, pending = k[k.payment_status == "Paid"], k[k.payment_status == "Refunded"], k[k.payment_status == "Pending"]
n_raw_rows = len(raw)

kpis = pd.DataFrame([
    ("K01", "Certified Orders",           len(k),                                                    "count"),
    ("K02", "Gross Order Value",          k.gross.sum(),                                             "INR"),
    ("K03", "Net Order Value",            k.net.sum(),                                               "INR"),
    ("K04", "Net Revenue (Paid)",         paid.net.sum(),                                            "INR"),
    ("K05", "Average Order Value (Paid)", paid.net.sum() / len(paid),                                "INR"),
    ("K06", "Payment Success Rate",       len(paid) / len(k),                                        "ratio"),
    ("K07", "Refund Rate",                len(refunded) / (len(paid) + len(refunded)),              "ratio"),
    ("K08", "Average Discount Rate",      1 - k.net.sum() / k.gross.sum(),                           "ratio"),
    ("K09", "Units per Order",            k.quantity.sum() / len(k),                                 "units"),
    ("K10", "Pending Value at Risk",      pending.net.sum(),                                         "INR"),
    ("K11", "Certified Row Rate",         len(k) / n_raw_rows,                                       "ratio"),
], columns=["kpi_id", "kpi", "value", "unit"])
display(kpis)

,kpi_id,kpi,value,unit
0,K01,Certified Orders,6.000000,count
1,K02,Gross Order Value,10790.000000,INR
2,K03,Net Order Value,10060.650000,INR
3,K04,Net Revenue (Paid),6013.350000,INR
4,K05,Average Order Value (Paid),1503.337500,INR
5,K06,Payment Success Rate,0.666667,ratio
6,K07,Refund Rate,0.200000,ratio
7,K08,Average Discount Rate,0.067595,ratio
8,K09,Units per Order,1.666667,units
9,K10,Pending Value at Risk,1499.000000,INR


## 7. Reconciliation and exports

In [8]:
# Every raw row must be accounted for exactly once.
n_dropped = int((remediation_log.action == "drop").sum())
assert len(raw) == len(certified) + len(quarantine) + n_dropped, "row reconciliation failed"
assert certified.order_id.is_unique, "certified layer must have unique order_id"
assert cert_results.loc[cert_results.severity == "critical", "status"].eq("PASS").all(), "critical check failing on certified layer"
print(f"Reconciled: {len(raw)} raw = {len(certified)} certified + {len(quarantine)} quarantined + {n_dropped} duplicate dropped")

raw_results.to_csv(f"{OUT_DIR}/dq_results_raw.csv", index=False)
cert_results.to_csv(f"{OUT_DIR}/dq_results_certified.csv", index=False)
certified.to_csv(f"{OUT_DIR}/certified_orders.csv", index=False)
quarantine.to_csv(f"{OUT_DIR}/quarantine.csv", index=False)
remediation_log.to_csv(f"{OUT_DIR}/remediation_log.csv", index=False)
kpis.to_csv(f"{OUT_DIR}/kpi_values.csv", index=False)
with open(f"{OUT_DIR}/gate.json", "w") as f:
    json.dump({"as_of": str(AS_OF.date()), "raw_gate": raw_gate, "certified_gate": cert_gate,
               "raw_rows": len(raw), "certified_rows": len(certified), "quarantined_rows": len(quarantine),
               "duplicates_dropped": n_dropped}, f, indent=2)
print("Exports written to", OUT_DIR)

Reconciled: 12 raw = 6 certified + 5 quarantined + 1 duplicate dropped
Exports written to profile_outputs


---
Documented by Rithi | rithicoded